# Description: Figure 06

In [1]:
import os.path as osp
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler 
from utils.basics import get_dataset_index
from utils.basics import PRCS_DATA_DIR
from tqdm.notebook import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
import hvplot.pandas
import holoviews as hv
import panel as pn
hv.extension('bokeh')
pn.extension('mathjax')

In [2]:
DATASET='evaluation'

In [3]:
ds_index = get_dataset_index(DATASET)
ses_list = list(ds_index.get_level_values('Session').unique())
sbj_list = list(ds_index.get_level_values('Subject').unique())

++ Number of scans    = 439
++ Number of subjects = 221


In [4]:
gs_kappa_rho_path = f'./summary_files/{DATASET}_gs_kappa_rho.csv'
gs_kappa_rho_df = pd.read_csv(gs_kappa_rho_path, index_col=[0,1])
print('++ INFO: GS Kappa and Rho estimates loaded from %s' % gs_kappa_rho_path)
print("++ INFO: The shape of kappa_rho_df is %s" % str(gs_kappa_rho_df.shape))
gs_kappa_rho_df.head(3)

++ INFO: GS Kappa and Rho estimates loaded from ./summary_files/evaluation_gs_kappa_rho.csv
++ INFO: The shape of kappa_rho_df is (439, 3)


kappa (GS)   rho (GS) kappa_rho_color
Subject Session                                       
sub-01  ses-1     40.671809  34.323632      lightgreen
        ses-2     33.884484  43.038005             red
sub-02  ses-1     35.800600  35.497794      lightgreen

In [5]:
real_varex_path = f'./summary_files/{DATASET}_varexp_gs_physio.real_data.csv'
real_varex_df   = pd.read_csv(real_varex_path, index_col=[0,1])
print('++ INFO: Estimates of variance explained from the GS by physio regressors loaded from %s' % real_varex_path)
print("++ INFO: The shape of real_varex_df is %s" % str(real_varex_df.shape))
real_varex_df.head(3)

++ INFO: Estimates of variance explained from the GS by physio regressors loaded from ./summary_files/evaluation_varexp_gs_physio.real_data.csv
++ INFO: The shape of real_varex_df is (370, 1)


Var. Exp. by Physio Regressors
Subject Session                                
sub-01  ses-1                          0.151980
        ses-2                          0.155122
sub-02  ses-1                          0.168487

In [6]:
null_varex_path = f'./summary_files/{DATASET}_varexp_gs_physio.null_distribution.csv'
null_varex_df   = pd.read_csv(null_varex_path, index_col=[0])
print('++ INFO: Null distribution for estimates of variance explained from the GS by physio regressors loaded from %s' % null_varex_path)
print("++ INFO: The shape of null_varex_df is %s" % str(null_varex_df.shape))
null_varex_df.head(3)

++ INFO: Null distribution for estimates of variance explained from the GS by physio regressors loaded from ./summary_files/evaluation_varexp_gs_physio.null_distribution.csv
++ INFO: The shape of null_varex_df is (10000, 1)


,Var. Exp. by Physio Regressors (NULL)
Permutation,
0,0.007868
1,0.045801
2,0.041574


In [7]:
mms = MinMaxScaler(feature_range=(2, 100))
motion_df = pd.DataFrame(index=ds_index, columns = ['Max. Motion (enorm)','Mean Motion (enorm)'])
for sbj,ses in tqdm(ds_index):
    mot_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'motion_{sbj}_enorm.1D')
    if osp.exists(mot_path):
        aux_mot = np.loadtxt(mot_path)
        motion_df.loc[(sbj,ses),'Mean Motion (enorm)'] = aux_mot.mean()
        motion_df.loc[(sbj,ses),'Max. Motion (enorm)'] = aux_mot.max()
motion_df['Mean Motion (dot size)'] = mms.fit_transform(motion_df['Mean Motion (enorm)'].values.reshape(-1,1))
motion_df['Max. Motion (dot size)'] = mms.fit_transform(motion_df['Max. Motion (enorm)'].values.reshape(-1,1))
motion_df = motion_df.infer_objects()
print (motion_df.shape)

  0%|          | 0/439 [00:00<?, ?it/s]

(439, 4)


# Panel a

In [8]:
df = pd.concat([gs_kappa_rho_df, motion_df], axis=1)
df.head(3)

kappa (GS)   rho (GS) kappa_rho_color  Max. Motion (enorm)  \
Subject Session                                                               
sub-01  ses-1     40.671809  34.323632      lightgreen             0.098541   
        ses-2     33.884484  43.038005             red             0.085322   
sub-02  ses-1     35.800600  35.497794      lightgreen             0.202130   

                 Mean Motion (enorm)  Mean Motion (dot size)  \
Subject Session                                                
sub-01  ses-1               0.026266                3.799107   
        ses-2               0.025437                3.528927   
sub-02  ses-1               0.031335                5.452014   

                 Max. Motion (dot size)  
Subject Session                          
sub-01  ses-1                  4.057597  
        ses-2                  3.347051  
sub-02  ses-1                  9.625603

In [9]:
# Set seaborn theme
sns.set_theme(style="white", context="notebook")

# Compute quantile limits for color normalization
vmin = df["Max. Motion (enorm)"].quantile(0.05)
vmax = df["Max. Motion (enorm)"].quantile(0.99)

# Create JointGrid
g = sns.JointGrid(data=df, x="kappa (GS)", y="rho (GS)", space=0, height=8)

# Define axes limits for the diagonal line
min_val = 0
max_val = 125

# Create diagonal line (y = x)
line_x = np.linspace(min_val, max_val, 500)
line_y = line_x

# Fill above (light red) and below (light green)
g.ax_joint.fill_between(line_x, line_y, max_val, color="#FFCCCC", alpha=0.4)  # Light red above
g.ax_joint.fill_between(line_x, min_val, line_y, color="#CCFFCC", alpha=0.4)  # Light green below

# Draw the diagonal line
g.ax_joint.plot(line_x, line_y, color="black", linestyle="--", linewidth=1)

#sizes = df["Max. Motion (dot size)"]
sc = g.ax_joint.scatter(
    data=df,
    x="kappa (GS)",
    y="rho (GS)",
    s=df['Max. Motion (dot size)'],
    c=df["Max. Motion (enorm)"],
    cmap="cividis",
    vmin=vmin, vmax=vmax,
    alpha=0.8,
    edgecolor="k",
    linewidth=0.3,
)

# Create new axes above the top marginal histogram
pos_joint = g.ax_joint.get_position()
pos_marg_x = g.ax_marg_x.get_position()
cbar_ax = g.fig.add_axes([pos_joint.x0, pos_marg_x.y1 + 0.05, pos_joint.width, 0.02])  # [left, bottom, width, height]

# Add horizontal colorbar
cbar = plt.colorbar(sc, cax=cbar_ax, orientation='horizontal')

# Move label and ticks to the top
cbar.ax.xaxis.set_label_position('top')
cbar.ax.xaxis.tick_top()
cbar.set_label("Max. Motion (mm)", rotation=0, labelpad=5)

# Top and right marginal plots
sns.histplot(x=df["kappa (GS)"], ax=g.ax_marg_x, kde=True, color="green", edgecolor="black", bins=50)
g.ax_marg_x.axvline(
    df["kappa (GS)"].median(), color="black", linewidth=3, linestyle='--', label='Median'
)
sns.histplot(y=df["rho (GS)"], ax=g.ax_marg_y, kde=True, color="red", edgecolor="black", bins=50)
g.ax_marg_y.axhline(
    df["rho (GS)"].median(), color='black', linewidth=3, linestyle='--', label='Median'
)
# Axis labels
g.set_axis_labels("kappa (GS)", "rho (GS)")

# Improve layout
g.ax_joint.set_xlim(12,125)
g.ax_joint.set_ylim(12,125)

# Save to disk
plt.tight_layout()
plt.close()
# After all plotting code (and before/without plt.show())
out_png = "./figures/pBOLD_Figure06_a.png"

g.figure.savefig( out_png, dpi=300, bbox_inches="tight",   # critical: captures cbar outside default figure bounds
    pad_inches=0.05)

print(f"Saved: {out_png}")

/tmp/ipykernel_3188332/4030524708.py:70: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: ./figures/pBOLD_Figure06_a.png


# Panel b

In [10]:
non_parametric_p005 = null_varex_df.quantile(0.95).values[0]
print('++ INFO: Non-parametric value for p=0.05 --> %.2f' % non_parametric_p005)

++ INFO: Non-parametric value for p=0.05 --> 0.12


In [14]:
x0, x1 = non_parametric_p005, 1.0
y_ann = 4.5
def add_span_label(plot, element):
    ax = plot.handles["axis"]
    xm = 0.5 * (x0 + x1)
    gap = 0.4 * (x1 - x0)  # small gap around text

    # Left and right arrows (so arrows are not part of the text string)
    ax.annotate(
        "", xy=(x0, y_ann), xytext=(xm - gap, y_ann),
        arrowprops=dict(arrowstyle="->", lw=1.8, color="black"),
        annotation_clip=False
    )
    ax.annotate(
        "", xy=(x1, y_ann), xytext=(xm + gap, y_ann),
        arrowprops=dict(arrowstyle="->", lw=1.8, color="black"),
        annotation_clip=False
    )

    ax.text(
        xm, y_ann, r"Statistically Significant Adjusted $R^{2}$ values",
        ha="center", va="center", fontsize=16, color="black",
        zorder=10
    )

In [15]:
hv.extension('matplotlib')
panel_b = (hv.VSpan(non_parametric_p005,1.0).opts(facecolor='lightgray', color='gray') * \
real_varex_df.hvplot.hist(bins=20, xlim=(0,1),                     ylabel=' % Scans', xlabel=r"$Adjusted R^{2}$",normed=True, label='Real Data', legend=False, fontscale=1.5, facecolor='gray') * \
real_varex_df.hvplot.kde(          xlim=(0,1),                     ylabel=' % Scans',              label='Real Data', legend=False, facecolor='gray'))
panel_b = panel_b.opts(hooks=[add_span_label])

<img src='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAYAAACqaXHeAAAABHNCSVQICAgIfAhkiAAAAAlwSFlz
AAAB+wAAAfsBxc2miwAAABl0RVh0U29mdHdhcmUAd3d3Lmlua3NjYXBlLm9yZ5vuPBoAAA6zSURB
VHic7ZtpeFRVmsf/5966taWqUlUJ2UioBBJiIBAwCZtog9IOgjqACsogKtqirT2ttt069nQ/zDzt
tI4+CrJIREFaFgWhBXpUNhHZQoKBkIUASchWla1S+3ar7r1nPkDaCAnZKoQP/D7mnPOe9/xy76n3
nFSAW9ziFoPFNED2LLK5wcyBDObkb8ZkxuaoSYlI6ZcOKq1eWFdedqNzGHQBk9RMEwFAASkk0Xw3
ETacDNi2vtvc7L0ROdw0AjoSotQVkKSvHQz/wRO1lScGModBFbDMaNRN1A4tUBCS3lk7BWhQkgpD
lG4852/+7DWr1R3uHAZVQDsbh6ZPN7CyxUrCzJMRouusj0ipRwD2uKm0Zn5d2dFwzX1TCGhnmdGo
G62Nna+isiUqhkzuKrkQaJlPEv5mFl2fvGg2t/VnzkEV8F5ioioOEWkLG86fvbpthynjdhXYZziQ
x1hC9J2NFyi8vCTt91Fh04KGip0AaG9zuCk2wQCVyoNU3Hjezee9bq92duzzTmxsRJoy+jEZZZYo
GTKJ6SJngdJqAfRzpze0+jHreUtPc7gpBLQnIYK6BYp/uGhw9YK688eu7v95ysgshcg9qSLMo3JC
4jqLKQFBgdKDPoQ+Pltb8dUyQLpeDjeVgI6EgLIQFT5tEl3rn2losHVsexbZ3EyT9wE1uGdkIPcy
BGxn8QUq1QrA5nqW5i2tLqvrrM9NK6AdkVIvL9E9bZL/oyfMVd/jqvc8LylzRBKDJSzIExwhQzuL
QYGQj4rHfFTc8mUdu3E7yoLtbTe9gI4EqVgVkug2i5+uXGo919ixbRog+3fTbQ8qJe4ZOYNfMoTI
OoshUNosgO60AisX15aeI2PSIp5KiFLI9ubb1vV3Qb2ltwLakUCDAkWX7/nHKRmmGIl9VgYsUhJm
2NXjKYADtM1ygne9QQDIXlk49FBstMKx66D1v4+XuQr7vqTe0VcBHQlRWiOCbmmSYe2SqtL6q5rJ
zsTb7lKx3FKOYC4DoqyS/B5bvLPxvD9Qtf6saxYLQGJErmDOdOMr/zo96km1nElr8bmPOBwI9COv
HnFPRIwmkSOv9kcAS4heRsidOkpeWBgZM+UBrTFAXNYL5Vf2ii9c1trNzpYdaoVil3WIc+wdk+gQ
noie3ecCcxt9ITcLAPWt/laGEO/9U6PmzZkenTtsSMQ8uYywJVW+grCstAvCIaAdArAsIWkRDDs/
KzLm2YcjY1Lv0UdW73HabE9n6V66cxSzfEmuJssTpKGVp+0vHq73FwL46eOjpMpbRAnNmJFrGJNu
Ukf9Yrz+3rghiumCKNXXWPhLYcjxGsIpoCMsIRoFITkW8AuyM8jC1+/QLx4bozCEJIq38+1rtpR6
V/yzb8eBlRb3fo5l783N0CWolAzJHaVNzkrTzlEp2bQ2q3TC5gn6wpnoQAmwSiGh2GitnTmVMc5O
UyfKWUKCIsU7+fZDKwqdT6DDpvkzAX4/+AMFjk0tDp5GRXLpQ2MUmhgDp5gxQT8+Y7hyPsMi8uxF
71H0oebujHALECjFKaW9Lm68n18wXp2kVzIcABytD5iXFzg+WVXkegpAsOOYziqo0OkK76GyquC3
ltZAzMhhqlSNmmWTE5T6e3IN05ITFLM4GdN0vtZ3ob8Jh1NAKXFbm5PtLU/eqTSlGjkNAJjdgn/N
aedXa0tdi7+t9G0FIF49rtMSEgAs1kDLkTPO7ebm4IUWeyh1bKomXqlgMG6kJmHcSM0clYLJ8XtR
1GTnbV3F6I5wCGikAb402npp1h1s7LQUZZSMIfALFOuL3UUrfnS8+rez7v9qcold5tilgHbO1fjK
9ubb17u9oshxzMiUBKXWqJNxd+fqb0tLVs4lILFnK71H0Ind7uiPgACVcFJlrb0tV6DzxqqTIhUM
CwDf1/rrVhTa33/3pGPxJYdQ2l2cbgVcQSosdx8uqnDtbGjh9SlDVSMNWhlnilfqZk42Th2ZpLpf
xrHec5e815zrr0dfBZSwzkZfqsv+1FS1KUknUwPARVvItfKUY+cn57yP7qv07UE3p8B2uhUwLk09
e0SCOrK+hbdYHYLjRIl71wWzv9jpEoeOHhGRrJAzyEyNiJuUqX0g2sBN5kGK6y2Blp5M3lsB9Qh4
y2Ja6x6+i0ucmKgwMATwhSjdUu49tKrQ/pvN5d53ml2CGwCmJipmKjgmyuaXzNeL2a0AkQ01Th5j
2DktO3Jyk8f9vcOBQHV94OK+fPumJmvQHxJoWkaKWq9Vs+yUsbq0zGT1I4RgeH2b5wef7+c7bl8F
eKgoHVVZa8ZPEORzR6sT1BzDUAD/d9F78e2Tzv99v8D+fLVTqAKAsbGamKey1Mt9Ann4eH3gTXTz
idWtAJ8PQWOk7NzSeQn/OTHDuEikVF1R4z8BQCy+6D1aWRfY0tTGG2OM8rRoPaeIj5ZHzJxszElN
VM8K8JS5WOfv8mzRnQAKoEhmt8gyPM4lU9SmBK1MCQBnW4KONT86v1hZ1PbwSXPw4JWussVjtH9Y
NCoiL9UoH/6PSu8jFrfY2t36erQHXLIEakMi1SydmzB31h3GGXFDFNPaK8Rme9B79Ixrd0WN+1ij
NRQ/doRmuFLBkHSTOm5GruG+pFjFdAmorG4IXH1Qua6ASniclfFtDYt+oUjKipPrCQB7QBQ2lrgP
fFzm+9XWUtcqJ3/5vDLDpJ79XHZk3u8nGZ42qlj1+ydtbxysCezrydp6ugmipNJ7WBPB5tydY0jP
HaVNzs3QzeE4ZpTbI+ZbnSFPbVOw9vsfnVvqWnirPyCNGD08IlqtYkh2hjZ5dErEQzoNm+6ykyOt
Lt5/PQEuSRRKo22VkydK+vvS1XEKlhCJAnsqvcVvH7f/ZU2R67eXbMEGAMiIV5oWZWiWvz5Fv2xG
sjqNJQRvn3Rs2lji/lNP19VjAQDgD7FHhujZB9OGqYxRkZxixgRDVlqS6uEOFaJUVu0rPFzctrnF
JqijImVp8dEKVWyUXDk92zAuMZ6bFwpBU1HrOw6AdhQgUooChb0+ItMbWJitSo5Ws3IAOGEOtL53
0vHZih9sC4vtofZ7Qu6523V/fmGcds1TY3V36pUsBwAbSlxnVh2xLfAD/IAIMDf7XYIkNmXfpp2l
18rkAJAy9HKFaIr/qULkeQQKy9zf1JgDB2uaeFNGijo5QsUyacNUUTOnGO42xSnv4oOwpDi1zYkc
efUc3I5Gk6PhyTuVKaOGyLUAYPGIoY9Pu/atL/L92+4q9wbflRJ2Trpm/jPjdBtfnqB/dIThcl8A
KG7hbRuKnb8qsQsVvVlTrwQAQMUlf3kwJI24Z4JhPMtcfng5GcH49GsrxJpGvvHIaeem2ma+KSjQ
lIwUdYyCY8j4dE1KzijNnIP2llF2wcXNnsoapw9XxsgYAl6k+KzUXbi2yP3KR2ecf6z3BFsBICdW
nvnIaG3eHybqX7vbpEqUMT+9OL4Qpe8VON7dXuFd39v19FoAABRVePbGGuXTszO0P7tu6lghUonE
llRdrhArLvmKdh9u29jcFiRRkfLUxBiFNiqSU9icoZQHo5mYBI1MBgBH6wMNb+U7Pnw337H4gi1Y
ciWs+uks3Z9fztUvfzxTm9Ne8XXkvQLHNytOOZeiD4e0PgkAIAYCYknKUNUDSXEKzdWNpnil7r4p
xqkjTarZMtk/K8TQ6Qve78qqvXurGwIJqcOUKfUWHsm8KGvxSP68YudXq4pcj39X49uOK2X142O0
Tz5/u/7TVybqH0rSya6ZBwD21/gubbrgWdDgEOx9W

# Put together

In [16]:
pn.Column( pn.Row(pn.pane.Markdown('## (a)'), width=10),
                  pn.pane.PNG('./figures/pBOLD_Figure06_a.png', width=700),
           pn.Row(pn.pane.Markdown('## (b)'), width=10),
                  pn.panel(panel_b, width=700)         
        ).save('./figures/pBOLD_Figure06.html')

/data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/holoviews/plotting/mpl/annotation.py:126: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  return [axis.axvspan(*positions, **opts)]


Here is the final figure

![Figure 06](figures/pBOLD_Figure06.png)